![](Img/img1.jpg)

# 🖼️ TorchVision → Computer Vision

## 🔑 Applications
- **Datasets**: CIFAR-10, MNIST  
- **Pretrained CNNs**  
- **Utilities for Image Transformations**

---

## 📊 Image Preprocessing Workflow

**Image → Scale → Normalize**

| Step        | Transformation | Range Conversion |
|-------------|----------------|------------------|
| Raw Pixels  | Scale          | (0 – 255) → (0 – 1) |
| Scaled Data | Normalize      | (0 – 1) → (-1 – 1) |

---

✨ This format makes it easy to follow the pipeline and highlights TorchVision’s core applications.


![](Img/img2.png)

In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [39]:
# Datasets & DataLoader
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms
# **Image → Scale(0, 1) → Normalize(-1, +1) **
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        #                 Standard Deviation,   Mean  
        transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
    ]
)
 # Load dataset (CIFAR-10 here)
trainset = CIFAR10(root= "./data", train= True, download= True, transform= transform)
testset = CIFAR10(root= "./data", train= False, download= True, transform= transform)

In [40]:
# Data Loader
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

# Build The CNN

![](Img/img3.png)

In [41]:
# This image represents it is done after the #  CONVOLUTIONAL LAYER AND BEFORE FC LAYER

In [42]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
#  CONVOLUTIONAL LAYER
        self.conv_layers = nn.Sequential(
            # 1ST Layer
    #Input channel(RGB)   Output Channel( no of filters :- it is multiple of two and double at every layer)
            #         ^   ^
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  # kernal size = 2 , stride = 2

            # 2nd Layer
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  # kernal size = 2 , stride = 2
            # 3rd Layer
            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  # kernal size = 2 , stride = 2
        )
# FULLY CONNECTED LAYER
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),
            
            nn.Linear(256, 10)
        )
    def forward( self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)  # Flattening step convert into 1D
        x = self.fc_layers(x)

        return x

In [43]:
# creating CNN Model 
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Train The CNN

In [44]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad() 
        
        output = model.forward(images)  # Forward Propopgation
        loss = criterion(output, labels) # loass fnx
        loss.backward()  # Backward propogation
        optimizer.step()  # update params

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch +1}/{epochs} & Loss ={ epoch_training_loss / len(trainloader)} ")

epoch = 1/10 & Loss =1.3625282264883867 
epoch = 2/10 & Loss =0.9123326707678987 
epoch = 3/10 & Loss =0.7248164068173875 
epoch = 4/10 & Loss =0.597642442111469 
epoch = 5/10 & Loss =0.49068548532245715 
epoch = 6/10 & Loss =0.4003419769008446 
epoch = 7/10 & Loss =0.308681318531637 
epoch = 8/10 & Loss =0.23842049446290411 
epoch = 9/10 & Loss =0.17624249694216282 
epoch = 10/10 & Loss =0.1429173643593593 


In [47]:
# Evalute 
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels/ total_labels * 100}")
print("Sample Size", total_labels)

Accuracy = 74.97
Sample Size 10000
